# Statsmodels: interpreting models and their uncertainty

A café wants to understand how a numerical outcome varies with observed predictors, including how uncertain those relationships are. Fit an interpretable model, translate coefficients into comparisons and inspect assumptions before relying on intervals or significance tests.

A prediction tells us what a model expects. Statistical modeling also asks how a response relates to explanatory variables, how uncertain the estimated relationships are, and whether the model's assumptions are plausible.

Imagine a café recording waiting times, party sizes, and service periods. We will fit a regression, interpret its coefficients, compare confidence and prediction intervals, and inspect residuals. Later examples introduce interactions and generalized linear models.

All café data below is **synthetic teaching data**. A separate example uses the small real Longley dataset bundled with Statsmodels. Everything runs offline after installation. Run cells from top to bottom.

## 1. Install and import

If needed, run `%pip install "statsmodels>=0.14" pandas numpy matplotlib` in a separate cell.

| Interface | What it accepts | Intercept behavior |
| --- | --- | --- |
| `statsmodels.formula.api` | A formula and a DataFrame | Includes an intercept by default |
| `statsmodels.api` | A response and an explicit design matrix | OLS requires you to add an intercept if wanted |

A model specifies the relationship. Calling `.fit()` returns a results object containing estimates, uncertainty, predictions, and diagnostics.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("Statsmodels:", statsmodels.__version__)

## 2. Build a reproducible example

The generator sets the mean waiting time to $5 + 2p + 4d$, where $p$ is party size and $d$ is 1 for dinner and 0 for lunch. Independent normally distributed noise adds variation with standard deviation 1.5 minutes.

Because we know the construction, we can compare estimates with the values 5, 2, and 4. Real observational data does not provide that privileged information.

In [ ]:
rng = np.random.default_rng(42)
n = 240
party_size = rng.integers(1, 6, size=n)
period = rng.choice(["Lunch", "Dinner"], size=n)
wait = 5 + 2 * party_size + 4 * (period == "Dinner") + rng.normal(0, 1.5, size=n)
cafe = pd.DataFrame({"party_size": party_size, "period": period, "wait_minutes": wait})
cafe["period"] = pd.Categorical(cafe["period"], categories=["Lunch", "Dinner"])
print(cafe.head())
print("Missing values:", cafe.isna().sum().to_dict())
print(cafe.groupby("period", observed=True)["wait_minutes"].agg(["size", "mean", "std"]))

## 3. Write the regression as a formula

The formula `wait_minutes ~ party_size + C(period)` models waiting time using party size and a categorical service-period effect. Lunch is the reference category because we ordered the categories explicitly.

Ordinary least squares (OLS) chooses coefficients that minimize the sum of squared residuals. Here a residual is observed waiting time minus fitted waiting time. We use all rows for this explanatory exercise; the resulting fit statistics are in-sample and do not measure future prediction performance.

In [ ]:
formula = "wait_minutes ~ party_size + C(period)"
ols = smf.ols(formula, data=cafe, missing="raise").fit()
print(ols.summary())

## 4. Translate a coefficient into a concrete comparison

Identify the reference category and units before interpreting a coefficient. Describe two otherwise comparable rows that differ by one unit of a predictor. The resulting contrast is conditional on the model, not automatically causal.

| Term | Interpretation in this additive model |
| --- | --- |
| Intercept | Expected lunch wait for a party of zero |
| `party_size` | Expected difference in minutes for one additional person, holding period fixed |
| `C(period)[T.Dinner]` | Expected dinner-minus-lunch difference, holding party size fixed |

The intercept uses a party size outside our observed range of 1–5; it is a mathematical anchor, not an observed waiting time. Standard errors quantify estimation uncertainty under the fitted model's assumptions.

A 95% confidence procedure has 95% coverage over repeated samples under its assumptions. A p-value evaluates compatibility with a zero-coefficient null; it is not the probability that the null is true, and it does not measure practical importance.

In [ ]:
intervals = ols.conf_int(alpha=0.05)
coefficient_table = pd.DataFrame({
    "estimate": ols.params,
    "standard_error": ols.bse,
    "p_value": ols.pvalues,
    "ci_low": intervals[0],
    "ci_high": intervals[1],
})
print(coefficient_table.round(4))
print(f"In-sample R²: {ols.rsquared:.3f}")
print(f"Adjusted R²: {ols.rsquared_adj:.3f}")

## 5. Center a predictor to make the intercept useful

Subtract 3 from party size. The new intercept describes a three-person lunch party. Centering changes this reference point, but the fitted predictions and party-size slope stay the same.

This is a change in how we express the model, not additional evidence about the relationship.

In [ ]:
cafe["party_centered"] = cafe["party_size"] - 3
centered = smf.ols("wait_minutes ~ party_centered + C(period)", data=cafe, missing="raise").fit()
print(centered.params)
print("Maximum fitted-value difference:", np.max(np.abs(ols.fittedvalues - centered.fittedvalues)))

## 6. Compare the formula and matrix interfaces

The matrix interface requires numerical columns, including a dinner indicator and an explicit constant. Both interfaces fit the same model here.

A design matrix contains the actual columns used by regression. Formula syntax constructs those columns for us, including categorical indicators and interactions.

In [ ]:
design = pd.DataFrame({"party_size": cafe["party_size"],
                       "dinner": (cafe["period"] == "Dinner").astype(float)})
design = sm.add_constant(design)
matrix_fit = sm.OLS(cafe["wait_minutes"], design, missing="raise").fit()
print(matrix_fit.params)
print("Same fitted values:", np.allclose(matrix_fit.fittedvalues, ols.fittedvalues))

## 7. Choose an interval for the quantity being predicted

Predict which interval should be wider: uncertainty about a mean response or uncertainty about a new individual observation. Compare them at the same predictor values and explain the extra variation in the latter.

For each new row, `mean_ci_lower` and `mean_ci_upper` describe uncertainty about the conditional mean response. `obs_ci_lower` and `obs_ci_upper` describe an interval for one new observation and include residual variation, so they are wider.

These ordinary OLS intervals use the model's constant-variance and distributional assumptions. Staying within observed party sizes also avoids extrapolating beyond the fitted range.

In [ ]:
new_parties = pd.DataFrame({"party_size": [1, 2, 3, 4, 5], "period": ["Lunch"] * 5})
prediction_intervals = ols.get_prediction(new_parties).summary_frame(alpha=0.05)
print(pd.concat([new_parties, prediction_intervals], axis=1).round(3))
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
x = new_parties["party_size"].to_numpy()
ax.fill_between(x, prediction_intervals["obs_ci_lower"].to_numpy(), prediction_intervals["obs_ci_upper"].to_numpy(),
                alpha=0.2, color="tab:blue", label="95% prediction interval")
ax.fill_between(x, prediction_intervals["mean_ci_lower"].to_numpy(), prediction_intervals["mean_ci_upper"].to_numpy(),
                alpha=0.5, color="tab:blue", label="95% mean confidence interval")
ax.plot(x, prediction_intervals["mean"], marker="o", color="black", label="Fitted lunch mean")
ax.set(xlabel="Party size", ylabel="Wait (minutes)", title="Mean uncertainty vs individual variation")
ax.set_xticks(x)
ax.legend()
plt.show()

## 8. Look for structure the fitted model has missed

Before interpreting p-values, inspect residuals for curvature, changing spread and unusual observations. A pattern suggests a modeling assumption worth revisiting; no obvious pattern is not proof that every assumption holds.

A residual-versus-fitted plot can reveal curvature or changing spread. A normal Q–Q plot compares ordered standardized residuals with normal quantiles; strong departures may suggest heavy tails or other mismatch.

No single plot proves the assumptions. Independence also depends on how observations were collected: repeated orders from one customer or shared shifts may create dependence that these plots do not resolve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].scatter(ols.fittedvalues, ols.resid, alpha=0.5)
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set(xlabel="Fitted wait (minutes)", ylabel="Residual (minutes)", title="Residuals vs fitted values")
sm.qqplot(ols.resid, fit=True, line="45", ax=axes[1])
axes[1].set_title("Normal Q–Q plot of standardized residuals")
plt.show()

## 9. Test changing variance and compare robust standard errors

The Breusch–Pagan test assesses a constant-variance null against variance related to the supplied predictors. A large p-value does not prove constant variance, especially with limited data.

HC3 adjusts coefficient uncertainty for heteroskedasticity while leaving the OLS coefficients unchanged. It does not repair an incorrect mean model, dependence between observations, omitted variables, or causal identification. The ordinary prediction intervals above are not made heteroskedasticity-aware by fitting this separate result.

In [ ]:
lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(ols.resid, ols.model.exog)
print(f"Breusch–Pagan LM statistic: {lm_stat:.3f}; p-value: {lm_pvalue:.3f}")
robust = smf.ols(formula, data=cafe, missing="raise").fit(cov_type="HC3")
print(pd.DataFrame({"estimate": ols.params, "OLS_standard_error": ols.bse,
                    "HC3_standard_error": robust.bse}).round(4))

## 10. Check leverage and influence

Leverage describes how unusual an observation's predictors are relative to the design. Cook's distance combines residual size and leverage to measure influence on the fitted coefficients.

Inspect influential rows for recording problems or scientifically meaningful differences. Ranking influence is not a rule for deleting inconvenient observations.

In [ ]:
influence = ols.get_influence().summary_frame()
top_influence = influence.nlargest(5, "cooks_d")[["hat_diag", "student_resid", "cooks_d"]]
print(cafe.join(top_influence, how="inner").round(4))

## 11. Interactions allow slopes to differ

`party_size * C(period)` expands to the two main effects plus their interaction. The party-size coefficient is then the lunch slope; the interaction is the dinner-minus-lunch slope difference.

The generator used equal slopes, so the additive model matches its construction. We fit the interaction to learn the syntax. The nested-model F-test compares these models on the same rows under the usual OLS assumptions; a small p-value is not a substitute for a reason to include the term.

In [ ]:
interaction = smf.ols("wait_minutes ~ party_size * C(period)", data=cafe, missing="raise").fit()
print(interaction.params.round(3))
print(sm.stats.anova_lm(ols, interaction))
print(pd.DataFrame({"model": ["Additive", "Interaction"], "AIC": [ols.aic, interaction.aic],
                    "BIC": [ols.bic, interaction.bic]}).round(3))

## 12. Check whether predictors can be distinguished from one another

Predict what happens to coefficient uncertainty when two predictors carry almost the same information. Compare uncertainty and diagnostics rather than concluding that an unstable coefficient means the feature is irrelevant.

A variance inflation factor (VIF) describes how much a coefficient's variance is inflated by linear relationships among predictors. It is a diagnostic, not an automatic feature-deletion threshold. Scaling and substantive meaning also matter when interpreting a design.

We report predictor VIFs while retaining the intercept in each auxiliary regression.

In [ ]:
vif_table = pd.DataFrame([
    {"term": name, "VIF": variance_inflation_factor(ols.model.exog, index)}
    for index, name in enumerate(ols.model.exog_names) if name != "Intercept"
])
print(vif_table.round(3))

## 13. A binary response needs a different model

Generalized linear models (GLMs) connect a mean response to predictors using a link function and a response distribution. A binomial GLM with its default logit link models probabilities between zero and one.

Generate an independent synthetic yes/no outcome for each café row. `exp(coefficient)` is an odds ratio for a one-unit predictor difference with other predictors fixed. Odds are $p/(1-p)$; an odds ratio is not the same as a probability ratio. These fitted probabilities are in-sample illustrations, not held-out performance estimates.

In [ ]:
binary_rng = np.random.default_rng(7)
linear_score = -2 + 0.35 * cafe["party_size"] + 0.7 * (cafe["period"] == "Dinner")
true_probability = 1 / (1 + np.exp(-linear_score))
cafe["orders_dessert"] = binary_rng.binomial(1, true_probability)
logit = smf.glm("orders_dessert ~ party_size + C(period)", data=cafe,
                family=sm.families.Binomial(), missing="raise").fit()
print(pd.DataFrame({"log_odds_coefficient": logit.params, "odds_ratio": np.exp(logit.params)}).round(3))
dessert_probabilities = logit.predict(new_parties)
print("Estimated lunch dessert probabilities:", dessert_probabilities.round(3).tolist())

## 14. Count responses and the Poisson GLM

The default Poisson link is logarithmic, so exponentiating a coefficient gives a multiplicative change in the conditional expected count. We generate counts with equal exposure: each row represents one observation interval of the same length.

A Poisson model assumes conditional variance equals conditional mean. The Pearson chi-square statistic divided by residual degrees of freedom is a rough dispersion diagnostic; large values can motivate investigating overdispersion. Unequal observation durations require exposure or an appropriate log-offset.

In [ ]:
count_rng = np.random.default_rng(19)
expected_count = np.exp(0.2 + 0.2 * cafe["party_size"] + 0.3 * (cafe["period"] == "Dinner"))
cafe["extra_items"] = count_rng.poisson(expected_count)
poisson = smf.glm("extra_items ~ party_size + C(period)", data=cafe,
                  family=sm.families.Poisson(), missing="raise").fit()
print(pd.DataFrame({"coefficient": poisson.params, "mean_count_ratio": np.exp(poisson.params)}).round(3))
print(f"Pearson dispersion diagnostic: {poisson.pearson_chi2 / poisson.df_resid:.3f}")

## 15. Inspect a bundled real dataset

Longley is a small historical macroeconomic dataset often used to illustrate highly correlated predictors and numerical challenges. Read its packaged description and variable names before modeling.

These annual observations are time ordered and few in number. A regression summary alone would not justify causal or forecasting conclusions. This inspection exercise deliberately stops at understanding the data and its correlation structure.

In [ ]:
longley = sm.datasets.longley.load_pandas()
print(sm.datasets.longley.NOTE)
print("Dataset shape:", longley.data.shape)
print(longley.data.head())
print("Predictor correlations:")
print(longley.exog.corr().round(2))

## 16. Export a coefficient and diagnostic report

The left panel shows coefficients and their 95% confidence intervals for the additive café model. The right panel displays its residuals. These are model-based explanatory results on synthetic observations.

Running the cell creates `outputs` and writes PNG, SVG, CSV, and text files, replacing those named files if they exist.

In [ ]:
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)
report, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
positions = np.arange(len(coefficient_table))
errors = np.vstack([coefficient_table["estimate"] - coefficient_table["ci_low"],
                    coefficient_table["ci_high"] - coefficient_table["estimate"]])
axes[0].errorbar(coefficient_table["estimate"], positions, xerr=errors, fmt="o", capsize=4)
axes[0].set_yticks(positions, coefficient_table.index)
axes[0].axvline(0, color="gray", linestyle="--")
axes[0].set(title="OLS estimates and 95% confidence intervals", xlabel="Coefficient (response in minutes)")
axes[1].scatter(ols.fittedvalues, ols.resid, alpha=0.5)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(title="Residual check", xlabel="Fitted wait (minutes)", ylabel="Residual (minutes)")
report.suptitle("Café regression — synthetic teaching data")
for extension in ["png", "svg"]:
    destination = output_dir / f"statsmodels_cafe_report.{extension}"
    report.savefig(destination, dpi=150, bbox_inches="tight")
    print("Saved:", destination)
coefficient_table.to_csv(output_dir / "statsmodels_coefficients.csv", index_label="term")
(output_dir / "statsmodels_ols_summary.txt").write_text(ols.summary().as_text(), encoding="utf-8")
print("Saved coefficient CSV and model summary text.")
plt.show()

## 17. Practice and common mistakes

1. Double the generated noise standard deviation and compare interval widths.
2. Generate a genuine dinner-by-party-size interaction, then interpret both group slopes.
3. Make noise spread increase with party size. Compare residual plots and OLS/HC3 standard errors.
4. Add a nearly duplicate predictor and inspect coefficient uncertainty and VIFs.

| Observation | What to check |
| --- | --- |
| Unexpected intercept or category interpretation | Reference category, centering, and formula terms |
| Matrix OLS has no intercept | Add a constant explicitly when intended |
| Fewer observations than expected | Missing values and formula handling; use `missing="raise"` when appropriate |
| Small p-value treated as a large effect | Inspect units, coefficient magnitude, and interval width |
| High R² treated as causation | Study design, omitted variables, and temporal ordering |
| Robust errors treated as a complete repair | Mean specification and dependent observations still matter |
| Predictions for an unseen category fail | New data must match the categories understood by the formula |
| Many specifications tried until one is significant | Account for selection and multiple testing; plan substantive comparisons |

## Further reading

Use the official [linear regression documentation](https://www.statsmodels.org/stable/regression.html), [regression diagnostics example](https://www.statsmodels.org/stable/examples/notebooks/generated/regression_diagnostics.html), [GLM documentation](https://www.statsmodels.org/stable/glm.html), and [dataset descriptions](https://www.statsmodels.org/stable/datasets/index.html) to extend the notebook.

## Explain what the evidence supports

Interpret one coefficient and its interval in the example’s units. Identify a diagnostic that could weaken that interpretation and explain why statistical association alone does not justify an intervention.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: investigate a misleading operational association

A restaurant dataset suggests that an express service flag is associated with longer waits. Express service was mostly used on busy shifts, and waiting-time noise grows with party size. Clean duplicate and missing records, compare unadjusted and adjusted regression, and inspect conventional versus heteroscedasticity-robust uncertainty. The dataset is synthetic and demonstrates confounding; it does not estimate a real service's causal effect.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through a coefficient before interpreting significance

A difference in group means mixes the express flag with the kinds of shifts receiving it. If busy shifts both use express more and wait longer, an unadjusted coefficient can point in the opposite direction from an adjusted coefficient. Adjustment is a modeling assumption, not automatic causal identification: unobserved selection can remain.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

import statsmodels.formula.api as smf
p2_rng=np.random.default_rng(8);p2_n=600
p2_busy=p2_rng.binomial(1,.5,p2_n)
p2_express=p2_rng.binomial(1,np.where(p2_busy==1,.85,.15))
p2_party=p2_rng.integers(1,9,p2_n)
p2_wait=8+12*p2_busy+1.5*p2_party-3*p2_express+p2_rng.normal(size=p2_n)*(1+.6*p2_party)
p2_raw=pd.DataFrame({'record_id':np.arange(p2_n),'wait':p2_wait,'express':p2_express,'busy':p2_busy,'party':p2_party})
p2_raw.loc[0:9,'party']=np.nan
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-15:]],ignore_index=True)
p2_clean=p2_raw.drop_duplicates().dropna().copy()
assert len(p2_clean)==590
p2_naive=smf.ols('wait ~ express',data=p2_clean).fit()
p2_adjusted=smf.ols('wait ~ express + busy + party',data=p2_clean).fit()
p2_robust=smf.ols('wait ~ express + busy + party',data=p2_clean).fit(cov_type='HC3')
assert np.allclose(p2_adjusted.params,p2_robust.params)
assert p2_naive.params['express']>0 and p2_adjusted.params['express']<0

### Work through what robust standard errors actually change

HC3 changes estimated coefficient uncertainty under heteroscedasticity; it does not change the fitted OLS coefficients, remove omitted-variable bias or fix a wrong mean function. Compare the express coefficient and its interval across all three models. A narrower interval is not inherently a better model.

Prediction can be vectorized over a table rather than repeatedly invoking the formula machinery for each row. Compare exactly the same fitted model and input rows, so the performance comparison does not become a statistical-model comparison.

In [ ]:
p2_comparison=[]
for p2_name,p2_fit in [('unadjusted',p2_naive),('adjusted OLS',p2_adjusted),('adjusted HC3',p2_robust)]:
    p2_ci=p2_fit.conf_int().loc['express']
    p2_comparison.append({'model':p2_name,'express_coefficient':p2_fit.params['express'],'ci_low':p2_ci.iloc[0],'ci_high':p2_ci.iloc[1]})
p2_eval=p2_clean.iloc[:100]
p2_t=time.perf_counter();p2_vector=p2_adjusted.predict(p2_eval).to_numpy();p2_vector_time=time.perf_counter()-p2_t
p2_t=time.perf_counter();p2_rowwise=np.array([p2_adjusted.predict(p2_eval.iloc[[i]]).iloc[0] for i in range(len(p2_eval))]);p2_row_time=time.perf_counter()-p2_t
assert np.allclose(p2_vector,p2_rowwise)
p2_report=pd.DataFrame(p2_comparison);display(p2_report)
print({'batched_prediction_seconds':p2_vector_time,'rowwise_seconds':p2_row_time})
p2_report.to_csv(p2_out/'statsmodels_confounding_report.csv',index=False)

### Explain the reversal without overclaiming

The constructed data makes the busy-shift selection mechanism visible, so conditioning on busy and party size recovers a more relevant comparison. In observational reality, that mechanism would need substantive evidence. Missing party sizes were dropped; if missingness depends on waits or service type, complete-case estimates may be biased.

**Try:** omit the busy indicator again but keep HC3. **Worked reasoning:** the omitted-variable bias remains even though the standard errors change. Robust uncertainty is not a substitute for an appropriate comparison population. If repeated restaurants or shifts create correlated errors, heteroscedasticity-only correction is not the entire dependence model.